# Bash Tool-Calling Fine-Tuning with Unsloth

This notebook fine-tunes an 8B parameter LLM to perform bash tool calling using Unsloth for efficient training.

## Overview
- **Model**: Llama 3 8B (or similar 8B model)
- **Method**: QLoRA (4-bit quantization + LoRA)
- **Dataset**: 50K multi-turn bash tool-calling examples
- **Framework**: Unsloth + Hugging Face Transformers

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton
!pip install datasets wandb

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    # Model
    "model_name": "unsloth/llama-3-8b-bnb-4bit",  # 4-bit quantized Llama 3 8B
    "max_seq_length": 4096,  # Longer context for multi-turn
    "dtype": None,  # Auto-detect
    "load_in_4bit": True,
    
    # LoRA
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    
    # Training
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 100,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "fp16": not None,  # Will use bf16 if available
    "bf16": None,  # Auto-detect
    "logging_steps": 10,
    "save_steps": 500,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "seed": 42,
    
    # Data
    "train_file": "../data/processed/train_dataset.jsonl",
    "val_file": "../data/processed/val_dataset.jsonl",
    
    # Output
    "output_dir": "../outputs/bash-tool-calling-llama3-8b",
    "hub_model_id": None,  # Set to push to HuggingFace Hub
}

print("Configuration loaded!")
print(f"Model: {CONFIG['model_name']}")
print(f"Max sequence length: {CONFIG['max_seq_length']}")
print(f"LoRA rank: {CONFIG['lora_r']}")

## 3. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=CONFIG["dtype"],
    load_in_4bit=CONFIG["load_in_4bit"],
)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimization
    random_state=CONFIG["seed"],
    use_rslora=False,
    loftq_config=None,
)

# Print trainable parameters
model.print_trainable_parameters()

## 4. Prepare Dataset

In [ ]:
import json
from datasets import Dataset

def load_jsonl(filepath):
    """Load JSONL file into list of dicts."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_conversation(example):
    """
    Format a conversation into the chat template format.
    Handles tool calls specially.
    """
    messages = []
    
    for msg in example["conversations"]:
        role = msg["role"]
        content = msg.get("content", "")
        
        if role == "system":
            messages.append({"role": "system", "content": content})
        
        elif role == "user":
            messages.append({"role": "user", "content": content})
        
        elif role == "assistant":
            if msg.get("tool_calls"):
                # Format tool calls as structured text
                tool_calls = msg["tool_calls"]
                tool_text = "<tool_call>\n"
                for tc in tool_calls:
                    func = tc["function"]
                    tool_text += json.dumps({
                        "name": func["name"],
                        "arguments": json.loads(func["arguments"])
                    }, indent=2)
                tool_text += "\n</tool_call>"
                messages.append({"role": "assistant", "content": tool_text})
            else:
                messages.append({"role": "assistant", "content": content or ""})
        
        elif role == "tool":
            # Format tool results
            tool_content = f"<tool_response>\n{content}\n</tool_response>"
            messages.append({"role": "user", "content": tool_content})
    
    return messages

def prepare_dataset(filepath, tokenizer):
    """Load and prepare dataset for training."""
    raw_data = load_jsonl(filepath)
    print(f"Loaded {len(raw_data)} examples from {filepath}")
    
    formatted_data = []
    for example in raw_data:
        messages = format_conversation(example)
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        
        formatted_data.append({
            "text": text,
            "id": example["id"],
            "category": example.get("metadata", {}).get("category", "unknown")
        })
    
    return Dataset.from_list(formatted_data)

# Load datasets
print("Loading training dataset...")
train_dataset = prepare_dataset(CONFIG["train_file"], tokenizer)

print("\nLoading validation dataset...")
val_dataset = prepare_dataset(CONFIG["val_file"], tokenizer)

print(f"\nDataset sizes:")
print(f"  Training: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")

In [ ]:
# Preview a training example
print("Sample training example:")
print("=" * 50)
print(train_dataset[0]["text"][:2000])
print("...")

## 5. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Determine dtype
use_bf16 = is_bfloat16_supported()
print(f"Using bf16: {use_bf16}")

# Training arguments
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_steps=CONFIG["warmup_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    learning_rate=CONFIG["learning_rate"],
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    optim=CONFIG["optim"],
    weight_decay=CONFIG["weight_decay"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    seed=CONFIG["seed"],
    report_to="none",  # Set to "wandb" for logging
    evaluation_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=2,
    packing=False,  # Can enable for efficiency on shorter sequences
    args=training_args,
)

print("Trainer created!")
print(f"Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Reserved memory: {start_gpu_memory} GB")

In [ ]:
# Train!
print("Starting training...")
trainer_stats = trainer.train()

# Show final stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\nTraining complete!")
print(f"Peak memory used: {used_memory} GB ({used_percentage}%)")
print(f"Memory used for training: {used_memory_for_lora} GB")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

## 6. Save Model

In [ ]:
# Save LoRA adapters
lora_output_dir = f"{CONFIG['output_dir']}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"LoRA adapters saved to: {lora_output_dir}")

In [ ]:
# Optional: Save merged model (16-bit)
# This creates a full model that can be used without the base model

SAVE_MERGED = False  # Set to True to save merged model

if SAVE_MERGED:
    merged_output_dir = f"{CONFIG['output_dir']}/merged_16bit"
    model.save_pretrained_merged(
        merged_output_dir,
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"Merged model saved to: {merged_output_dir}")

In [ ]:
# Optional: Save in GGUF format for llama.cpp

SAVE_GGUF = False  # Set to True to save GGUF
GGUF_QUANT = "q4_k_m"  # Quantization method

if SAVE_GGUF:
    gguf_output_dir = f"{CONFIG['output_dir']}/gguf"
    model.save_pretrained_gguf(
        gguf_output_dir,
        tokenizer,
        quantization_method=GGUF_QUANT,
    )
    print(f"GGUF model saved to: {gguf_output_dir}")

## 7. Test the Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def generate_response(prompt, max_new_tokens=512):
    """Generate a response for a given prompt."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to a bash tool for executing commands on a Linux system. Use it to help users accomplish their tasks."},
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
# Test prompts
test_prompts = [
    "List all Python files in the current directory",
    "Find all files larger than 100MB",
    "Show me the git status",
    "Check disk space usage",
    "Search for 'TODO' in all source files",
]

print("Testing the fine-tuned model:")
print("=" * 60)

for prompt in test_prompts:
    print(f"\nUser: {prompt}")
    print("-" * 40)
    response = generate_response(prompt)
    # Extract just the assistant's response
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    print(f"Assistant: {response[:500]}..." if len(response) > 500 else f"Assistant: {response}")
    print()

## 8. Evaluation Metrics

In [ ]:
# Evaluate on validation set
eval_results = trainer.evaluate()

print("Evaluation Results:")
print("=" * 40)
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

In [ ]:
# Save training metrics
import json

metrics = {
    "training": trainer_stats.metrics,
    "evaluation": eval_results,
    "config": CONFIG,
}

with open(f"{CONFIG['output_dir']}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print(f"Metrics saved to: {CONFIG['output_dir']}/training_metrics.json")

## 9. Next Steps

After training, you can:

1. **Load the model for inference:**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "outputs/bash-tool-calling-llama3-8b/lora_adapters"
)
```

2. **Push to Hugging Face Hub:**
```python
model.push_to_hub("your-username/bash-tool-calling-llama3-8b")
tokenizer.push_to_hub("your-username/bash-tool-calling-llama3-8b")
```

3. **Run with llama.cpp** (if you saved GGUF):
```bash
./main -m outputs/gguf/model-q4_k_m.gguf -p "List all files"
```

4. **Fine-tune further** on domain-specific data

In [ ]:
print("Training complete! 🎉")
print(f"\nModel saved to: {CONFIG['output_dir']}")
print("\nContents:")
!ls -la {CONFIG['output_dir']}